In [1]:
# ============================================================
# STEP 120 — LOAD CTU-13 SCENARIO 2
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

SCENARIO2_FILE = Path(
    "../data/raw/CTU-13/scenario2/capture20110811.binetflow"
)

print("File exists:", SCENARIO2_FILE.exists())
print("File size (MB):", round(
    SCENARIO2_FILE.stat().st_size / (1024**2),
    2
))

scenario2_columns = [
    "StartTime",
    "Dur",
    "Proto",
    "SrcAddr",
    "Sport",
    "Dir",
    "DstAddr",
    "Dport",
    "TotPkts",
    "TotBytes",
    "SrcBytes",
    "Label"
]

c2_s2 = pd.read_csv(
    SCENARIO2_FILE,
    usecols=scenario2_columns
)

print("\nScenario 2 shape:", c2_s2.shape)
print("\nColumns:")
print(c2_s2.columns.tolist())

File exists: True
File size (MB): 235.78

Scenario 2 shape: (1808122, 12)

Columns:
['StartTime', 'Dur', 'Proto', 'SrcAddr', 'Sport', 'Dir', 'DstAddr', 'Dport', 'TotPkts', 'TotBytes', 'SrcBytes', 'Label']


In [2]:
# ============================================================
# STEP 121 — SCENARIO 2 LABEL ANALYSIS
# ============================================================

print("Total flows:", len(c2_s2))
print("Unique labels:", c2_s2["Label"].nunique())

print("\nTop labels:")
display(
    c2_s2["Label"]
    .value_counts()
    .head(40)
)

print("\nPotential C&C labels:")

c2_s2_cc = c2_s2[
    c2_s2["Label"]
    .astype(str)
    .str.contains(
        r"-CC\d+-",
        case=False,
        regex=True
    )
].copy()

print("Explicit C&C flows:", len(c2_s2_cc))

display(
    c2_s2_cc["Label"]
    .value_counts()
)

Total flows: 1808122
Unique labels: 132

Top labels:


Label
flow=To-Background-UDP-CVUT-DNS-Server             660177
flow=Background-UDP-Established                    602264
flow=Background-UDP-Attempt                        201380
flow=Background-TCP-Established                    149962
flow=Background-Established-cmpgw-CVUT              78133
flow=Background                                     29162
flow=Background-TCP-Attempt                         25929
flow=To-Background-CVUT-Proxy                       12950
flow=From-Botnet-V43-TCP-Attempt                     9759
flow=From-Normal-V43-Stribrek                        8960
flow=Background-Attempt-cmpgw-CVUT                   6096
flow=From-Botnet-V43-TCP-Attempt-SPAM                5477
flow=Background-UDP-NTP-Established-1                1601
flow=To-Background-CVUT-WebServer                    1449
flow=From-Botnet-V43-TCP-HTTP-Persistent-Down-1      1210
flow=From-Botnet-V43-UDP-DNS                          788
flow=Background-ajax.google                           734
flow=Bac


Potential C&C labels:
Explicit C&C flows: 673


Label
flow=From-Botnet-V43-TCP-CC69-Custom-Encryption           198
flow=From-Botnet-V43-TCP-CC66-HTTP-Custom-Encryption      182
flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted          114
flow=From-Botnet-V43-TCP-CC70-Custom-Encryption            89
flow=From-Botnet-V43-TCP-CC56-HTTP-Not-Encrypted           49
flow=From-Botnet-V43-TCP-CC1-HTTP-Not-Encrypted            30
flow=From-Botnet-V43-TCP-CC6-Plain-HTTP-Encrypted-Data     10
flow=From-Botnet-V43-TCP-CC7-Custom-Encryption              1
Name: count, dtype: int64

In [3]:
# ============================================================
# STEP 122 — SCENARIO 2 C&C ENDPOINT ANALYSIS
# ============================================================

c2_s2_cc = c2_s2[
    c2_s2["Label"]
    .astype(str)
    .str.contains(
        r"-CC\d+-",
        case=False,
        regex=True
    )
].copy()

c2_s2_cc["StartTime"] = pd.to_datetime(
    c2_s2_cc["StartTime"],
    errors="coerce"
)

c2_s2_cc["Dport_numeric"] = pd.to_numeric(
    c2_s2_cc["Dport"].astype(str).str.strip(),
    errors="coerce"
)

print("Explicit C&C flows:", len(c2_s2_cc))

print("\nC&C source addresses:")
display(
    c2_s2_cc["SrcAddr"]
    .value_counts()
    .head(20)
)

print("\nC&C destination addresses:")
display(
    c2_s2_cc["DstAddr"]
    .value_counts()
    .head(30)
)

print("\nC&C destination ports:")
display(
    c2_s2_cc["Dport_numeric"]
    .value_counts()
    .head(30)
)

Explicit C&C flows: 673

C&C source addresses:


SrcAddr
147.32.84.165    673
Name: count, dtype: int64


C&C destination addresses:


DstAddr
184.82.147.251     198
184.82.148.43      182
173.192.170.88     114
184.82.155.107      89
31.192.109.161      49
60.190.223.75       10
217.34.4.225         3
58.215.78.1          3
61.17.216.4          3
89.103.213.96        2
61.177.120.254       2
61.17.216.15         2
219.145.198.122      2
60.173.109.42        2
61.167.116.133       2
184.106.213.57       2
115.85.238.119       1
61.17.216.22         1
74.3.164.222         1
217.34.4.226         1
221.207.141.60       1
61.17.216.92         1
218.189.208.34       1
58.42.247.143        1
Name: count, dtype: int64


C&C destination ports:


Dport_numeric
80      632
6667     30
888      10
443       1
Name: count, dtype: int64

In [4]:
# ============================================================
# STEP 123 — DOMINANT C&C SOURCE
# ============================================================

c2_source_counts = (
    c2_s2_cc["SrcAddr"]
    .value_counts()
)

print("Top C&C sources:")
display(
    c2_source_counts.head(20)
)

print(
    "\nNumber of unique C&C source addresses:",
    c2_source_counts.shape[0]
)

Top C&C sources:


SrcAddr
147.32.84.165    673
Name: count, dtype: int64


Number of unique C&C source addresses: 1


In [5]:
# ============================================================
# STEP 124 — SCENARIO 2 C&C PAIR REPETITION
# ============================================================

s2_pairs = (
    c2_s2_cc
    .groupby(
        [
            "SrcAddr",
            "DstAddr",
            "Dport_numeric",
            "Proto"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

print("Repeated Scenario 2 C&C pairs:")

display(
    s2_pairs.head(30)
)

Repeated Scenario 2 C&C pairs:


SrcAddr        DstAddr          Dport_numeric  Proto
147.32.84.165  184.82.147.251   80             tcp      198
               184.82.148.43    80             tcp      182
               173.192.170.88   80             tcp      114
               184.82.155.107   80             tcp       89
               31.192.109.161   80             tcp       49
               60.190.223.75    888            tcp       10
               58.215.78.1      6667           tcp        3
               217.34.4.225     6667           tcp        3
               61.17.216.4      6667           tcp        3
               60.173.109.42    6667           tcp        2
               61.177.120.254   6667           tcp        2
               61.17.216.15     6667           tcp        2
               61.167.116.133   6667           tcp        2
               89.103.213.96    6667           tcp        2
               219.145.198.122  6667           tcp        2
               184.106.213.57   6667           

In [6]:
# ============================================================
# STEP 125 — SCENARIO 2 C&C INTER-ARRIVAL
# ============================================================

c2_s2_cc = (
    c2_s2_cc
    .sort_values(
        [
            "SrcAddr",
            "DstAddr",
            "Dport_numeric",
            "Proto",
            "StartTime"
        ]
    )
    .copy()
)

c2_s2_cc["iat_seconds"] = (
    c2_s2_cc
    .groupby(
        [
            "SrcAddr",
            "DstAddr",
            "Dport_numeric",
            "Proto"
        ]
    )["StartTime"]
    .diff()
    .dt.total_seconds()
)

print(
    "Valid C&C IAT measurements:",
    c2_s2_cc["iat_seconds"].notna().sum()
)

display(
    c2_s2_cc[
        [
            "StartTime",
            "SrcAddr",
            "DstAddr",
            "Dport_numeric",
            "Proto",
            "iat_seconds",
            "Label"
        ]
    ]
    .dropna(subset=["iat_seconds"])
    .head(30)
)

Valid C&C IAT measurements: 649


,StartTime,SrcAddr,DstAddr,Dport_numeric,Proto,iat_seconds,Label
458009,2011-08-11 10:41:51.105217,147.32.84.165,173.192.170.88,80,tcp,34.658969,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
463016,2011-08-11 10:42:31.487166,147.32.84.165,173.192.170.88,80,tcp,40.381949,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
492125,2011-08-11 10:46:16.674511,147.32.84.165,173.192.170.88,80,tcp,225.187345,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
497032,2011-08-11 10:46:59.178192,147.32.84.165,173.192.170.88,80,tcp,42.503681,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
504341,2011-08-11 10:48:01.229301,147.32.84.165,173.192.170.88,80,tcp,62.051109,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
528522,2011-08-11 10:51:23.836827,147.32.84.165,173.192.170.88,80,tcp,202.607526,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
534275,2011-08-11 10:52:05.009780,147.32.84.165,173.192.170.88,80,tcp,41.172953,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
543419,2011-08-11 10:53:06.294601,147.32.84.165,173.192.170.88,80,tcp,61.284821,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
569149,2011-08-11 10:56:18.408547,147.32.84.165,173.192.170.88,80,tcp,192.113946,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted
574601,2011-08-11 10:56:56.840393,147.32.84.165,173.192.170.88,80,tcp,38.431846,flow=From-Botnet-V43-TCP-CC16-HTTP-Not-Encrypted


In [7]:
# ============================================================
# STEP 126 — C&C TIMING SUMMARY
# ============================================================

s2_periodicity = (
    c2_s2_cc
    .groupby(
        [
            "SrcAddr",
            "DstAddr",
            "Dport_numeric",
            "Proto"
        ]
    )["iat_seconds"]
    .agg(
        flow_count="count",
        mean_iat="mean",
        std_iat="std",
        median_iat="median",
        min_iat="min",
        max_iat="max"
    )
    .reset_index()
)

s2_periodicity["iat_cv"] = (
    s2_periodicity["std_iat"]
    /
    s2_periodicity["mean_iat"]
)

s2_periodicity = (
    s2_periodicity
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["mean_iat"])
)

print("Scenario 2 C&C periodicity candidates:")

display(
    s2_periodicity
    .sort_values(
        ["flow_count", "iat_cv"],
        ascending=[False, True]
    )
    .head(30)
)

Scenario 2 C&C periodicity candidates:


,SrcAddr,DstAddr,Dport_numeric,Proto,flow_count,mean_iat,std_iat,median_iat,min_iat,max_iat,iat_cv
3,147.32.84.165,184.82.147.251,80,tcp,197,54.865452,252.832595,18.283692,8.740307,3215.218057,4.608229
4,147.32.84.165,184.82.148.43,80,tcp,181,64.046594,313.474193,20.484639,8.206925,4042.883860,4.894471
1,147.32.84.165,173.192.170.88,80,tcp,113,102.258216,115.394264,54.775029,0.000010,536.122707,1.128460
5,147.32.84.165,184.82.155.107,80,tcp,88,133.539463,308.792389,20.178225,8.227551,1614.502915,2.312368
11,147.32.84.165,31.192.109.161,80,tcp,48,244.901515,441.200878,63.974452,60.108695,2220.405261,1.801544
15,147.32.84.165,60.190.223.75,888,tcp,9,38.438997,101.181382,4.019758,1.089757,308.040939,2.632259
12,147.32.84.165,58.215.78.1,6667,tcp,2,1974.050027,347.379438,1974.050027,1728.415670,2219.684383,0.175973
6,147.32.84.165,217.34.4.225,6667,tcp,2,2331.044064,1467.006910,2331.044064,1293.713530,3368.374598,0.629335
19,147.32.84.165,61.17.216.4,6667,tcp,2,5359.607990,6785.619060,5359.607990,561.450738,10157.765242,1.266066
2,147.32.84.165,184.106.213.57,6667,tcp,1,1036.557561,NaN,1036.557561,1036.557561,1036.557561,NaN


In [8]:
# ============================================================
# STEP 127 — REUSABLE C2 FEATURE EXTRACTION
# ============================================================

def build_c2_window_features(
    df,
    window_seconds=5
):
    """
    Build source-centric temporal C2 features.

    Ground truth:
        Explicit labels containing -CC<number>-

    Features:
        Observable network behaviour only.
    """

    data = df.copy()

    # --------------------------------------------------------
    # Basic cleaning
    # --------------------------------------------------------

    data["StartTime"] = pd.to_datetime(
        data["StartTime"],
        errors="coerce"
    )

    data["Dport_numeric"] = pd.to_numeric(
        data["Dport"].astype(str).str.strip(),
        errors="coerce"
    )

    data = (
        data
        .dropna(
            subset=[
                "StartTime",
                "SrcAddr",
                "DstAddr"
            ]
        )
        .sort_values("StartTime")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # C&C ground truth
    # --------------------------------------------------------

    data["is_c2"] = (
        data["Label"]
        .astype(str)
        .str.contains(
            r"-CC\d+-",
            case=False,
            regex=True
        )
        .astype(int)
    )

    # --------------------------------------------------------
    # Time windows
    # --------------------------------------------------------

    data["time_window"] = (
        data["StartTime"]
        .dt.floor(f"{window_seconds}s")
    )

    # --------------------------------------------------------
    # Base window features
    # --------------------------------------------------------

    features = (
        data
        .groupby("time_window")
        .agg(
            flow_count=("StartTime", "size"),

            unique_destinations=(
                "DstAddr",
                "nunique"
            ),

            unique_ports=(
                "Dport_numeric",
                "nunique"
            ),

            total_packets=(
                "TotPkts",
                "sum"
            ),

            total_bytes=(
                "TotBytes",
                "sum"
            ),

            mean_packets=(
                "TotPkts",
                "mean"
            ),

            mean_bytes=(
                "TotBytes",
                "mean"
            ),

            mean_duration=(
                "Dur",
                "mean"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Destination concentration
    # --------------------------------------------------------

    destination_counts = (
        data
        .groupby(
            [
                "time_window",
                "DstAddr"
            ]
        )
        .size()
        .rename("destination_count")
        .reset_index()
    )

    destination_max = (
        destination_counts
        .groupby("time_window")
        ["destination_count"]
        .max()
        .rename("max_destination_count")
        .reset_index()
    )

    features = features.merge(
        destination_max,
        on="time_window",
        how="left"
    )

    features["destination_concentration"] = (
        features["max_destination_count"]
        /
        features["flow_count"]
    )

    # --------------------------------------------------------
    # Pair-level inter-arrival time
    # --------------------------------------------------------

    data = data.sort_values(
        [
            "DstAddr",
            "Dport_numeric",
            "Proto",
            "StartTime"
        ]
    )

    data["pair_iat"] = (
        data
        .groupby(
            [
                "DstAddr",
                "Dport_numeric",
                "Proto"
            ]
        )["StartTime"]
        .diff()
        .dt.total_seconds()
    )

    iat_features = (
        data
        .groupby("time_window")["pair_iat"]
        .agg(
            iat_mean="mean",
            iat_std="std",
            iat_median="median",
            iat_min="min",
            iat_max="max"
        )
        .reset_index()
    )

    iat_features["iat_cv"] = (
        iat_features["iat_std"]
        /
        iat_features["iat_mean"]
    )

    features = features.merge(
        iat_features,
        on="time_window",
        how="left"
    )

    # --------------------------------------------------------
    # Pair repetition
    # --------------------------------------------------------

    pair_counts = (
        data
        .groupby(
            [
                "time_window",
                "DstAddr",
                "Dport_numeric"
            ]
        )
        .size()
        .rename("pair_count")
        .reset_index()
    )

    pair_max = (
        pair_counts
        .groupby("time_window")
        ["pair_count"]
        .max()
        .rename("max_pair_repetition")
        .reset_index()
    )

    features = features.merge(
        pair_max,
        on="time_window",
        how="left"
    )

    features["pair_repetition_ratio"] = (
        features["max_pair_repetition"]
        /
        features["flow_count"]
    )

    # --------------------------------------------------------
    # Destination entropy
    # --------------------------------------------------------

    def destination_entropy(group):

        counts = group["DstAddr"].value_counts()

        probabilities = (
            counts / counts.sum()
        )

        return float(
            -(
                probabilities *
                np.log2(probabilities)
            ).sum()
        )

    entropy = (
        data
        .groupby("time_window")
        .apply(
            destination_entropy,
            include_groups=False
        )
        .rename("destination_entropy")
        .reset_index()
    )

    features = features.merge(
        entropy,
        on="time_window",
        how="left"
    )

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    target = (
        data
        .groupby("time_window")["is_c2"]
        .max()
        .rename("c2_target")
        .reset_index()
    )

    features = features.merge(
        target,
        on="time_window",
        how="left"
    )

    # --------------------------------------------------------
    # Final cleaning
    # --------------------------------------------------------

    features = (
        features
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .sort_values("time_window")
        .reset_index(drop=True)
    )

    return features

In [9]:
# ============================================================
# STEP 128 — BUILD SCENARIO 2 C2 FEATURES
# ============================================================

scenario2_c2_features = build_c2_window_features(
    c2_s2,
    window_seconds=5
)

print(
    "Scenario 2 feature table:",
    scenario2_c2_features.shape
)

print("\nScenario 2 target:")
print(
    scenario2_c2_features[
        "c2_target"
    ].value_counts()
)

display(
    scenario2_c2_features.head(10)
)

Scenario 2 feature table: (3020, 21)

Scenario 2 target:
c2_target
0    2434
1     586
Name: count, dtype: int64


,time_window,flow_count,unique_destinations,unique_ports,total_packets,total_bytes,mean_packets,mean_bytes,mean_duration,max_destination_count,...,iat_mean,iat_std,iat_median,iat_min,iat_max,iat_cv,max_pair_repetition,pair_repetition_ratio,destination_entropy,c2_target
0,2011-08-11 09:49:35,1078,326,280,538753,292067789,499.770872,270934.869202,1507.603828,500,...,0.071254,0.303259,0.006862,0.000003,4.061223,4.256047,443,0.410946,4.449647,0
1,2011-08-11 09:49:40,1146,354,323,144826,24902736,126.375218,21730.136126,1526.114854,515,...,0.175668,0.757121,0.008071,0.000005,7.067323,4.309940,470,0.410122,4.447752,0
2,2011-08-11 09:49:45,1093,301,234,43085,7363689,39.419030,6737.135407,1299.331215,462,...,0.337444,1.466690,0.008403,0.000003,12.852226,4.346465,445,0.407136,4.480556,0
3,2011-08-11 09:49:50,1052,321,279,64045,9774167,60.879278,9291.033270,1418.621748,468,...,0.423064,1.885936,0.008791,0.000004,16.998245,4.457800,446,0.423954,4.469947,0
4,2011-08-11 09:49:55,1048,339,285,42873,7283673,40.909351,6950.069656,1416.977836,463,...,0.474851,2.288188,0.007769,0.000006,20.880751,4.818746,440,0.419847,4.484200,0
5,2011-08-11 09:50:00,1111,291,220,58804,18616778,52.928893,16756.775878,1324.024657,482,...,0.625890,2.433276,0.007005,0.000006,21.358800,3.887707,458,0.412241,4.396425,0
6,2011-08-11 09:50:05,1005,302,259,30623,4998844,30.470647,4973.974129,1297.776384,445,...,0.672905,3.243406,0.009060,0.000005,30.510228,4.820009,418,0.415920,4.456047,0
7,2011-08-11 09:50:10,989,242,206,28887,4093129,29.208291,4138.654196,1310.583774,461,...,0.838480,3.823243,0.006482,0.000004,30.738506,4.559728,446,0.450961,3.660446,0
8,2011-08-11 09:50:15,915,273,221,53052,23468715,57.980328,25648.868852,1381.154254,410,...,1.049747,4.718641,0.008242,0.000005,42.418460,4.495026,393,0.429508,4.303017,0
9,2011-08-11 09:50:20,960,248,192,35123,14958656,36.586458,15581.933333,1259.053574,430,...,1.203857,4.950016,0.007768,0.000003,46.972461,4.111799,412,0.429167,4.089896,0


In [10]:
# ============================================================
# STEP 129 — LOAD SCENARIO 1
# ============================================================

SCENARIO1_FILE = Path(
    "../data/raw/CTU-13/capture20110810.binetflow"
)

scenario1_columns = [
    "StartTime",
    "Dur",
    "Proto",
    "SrcAddr",
    "Sport",
    "Dir",
    "DstAddr",
    "Dport",
    "TotPkts",
    "TotBytes",
    "SrcBytes",
    "Label"
]

c2_s1 = pd.read_csv(
    SCENARIO1_FILE,
    usecols=scenario1_columns
)

print(
    "Scenario 1 shape:",
    c2_s1.shape
)

Scenario 1 shape: (2824636, 12)


In [11]:
# ============================================================
# STEP 130 — BUILD SCENARIO 1 C2 FEATURES
# ============================================================

scenario1_c2_features = build_c2_window_features(
    c2_s1,
    window_seconds=5
)

print(
    "Scenario 1 feature table:",
    scenario1_c2_features.shape
)

print("\nScenario 1 target:")
print(
    scenario1_c2_features[
        "c2_target"
    ].value_counts()
)

Scenario 1 feature table: (4408, 21)

Scenario 1 target:
c2_target
0    4102
1     306
Name: count, dtype: int64


In [12]:
# ============================================================
# STEP 131 — SCENARIO IDENTIFIERS
# ============================================================

scenario1_c2_features["scenario"] = "CTU13_42"
scenario2_c2_features["scenario"] = "CTU13_43"

print(
    scenario1_c2_features["scenario"].unique()
)

print(
    scenario2_c2_features["scenario"].unique()
)

['CTU13_42']
['CTU13_43']


In [13]:
# ============================================================
# STEP 132 — MULTI-SCENARIO SUMMARY
# ============================================================

for name, data in [
    ("Scenario 1", scenario1_c2_features),
    ("Scenario 2", scenario2_c2_features)
]:

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Shape:", data.shape)

    print("\nTarget:")
    print(
        data["c2_target"].value_counts()
    )

    print(
        "\nC2 percentage:",
        round(
            data["c2_target"].mean() * 100,
            2
        )
    )


Scenario 1
Shape: (4408, 22)

Target:
c2_target
0    4102
1     306
Name: count, dtype: int64

C2 percentage: 6.94

Scenario 2
Shape: (3020, 22)

Target:
c2_target
0    2434
1     586
Name: count, dtype: int64

C2 percentage: 19.4


In [14]:
# ============================================================
# STEP 133 — COMMON C2 FEATURE SPACE
# ============================================================

# Remove scenario labels from the ML feature space.
# Both scenarios were produced by the same extractor.

EXCLUDE_COLUMNS = [
    "time_window",
    "c2_target",
    "scenario"
]

common_features = sorted(
    list(
        (
            set(scenario1_c2_features.columns)
            & set(scenario2_c2_features.columns)
        )
        - set(EXCLUDE_COLUMNS)
    )
)

print("Common feature count:", len(common_features))

print("\nCommon features:")
for col in common_features:
    print("-", col)

Common feature count: 19

Common features:
- destination_concentration
- destination_entropy
- flow_count
- iat_cv
- iat_max
- iat_mean
- iat_median
- iat_min
- iat_std
- max_destination_count
- max_pair_repetition
- mean_bytes
- mean_duration
- mean_packets
- pair_repetition_ratio
- total_bytes
- total_packets
- unique_destinations
- unique_ports


In [15]:
# ============================================================
# STEP 134 — CLEAN MULTI-SCENARIO DATA
# ============================================================

s1 = (
    scenario1_c2_features
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

s2 = (
    scenario2_c2_features
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

s1 = s1.replace(
    [np.inf, -np.inf],
    np.nan
)

s2 = s2.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Scenario 1:", s1.shape)
print("Scenario 2:", s2.shape)

print("\nScenario 1 target:")
print(s1["c2_target"].value_counts())

print("\nScenario 2 target:")
print(s2["c2_target"].value_counts())

Scenario 1: (4408, 22)
Scenario 2: (3020, 22)

Scenario 1 target:
c2_target
0    4102
1     306
Name: count, dtype: int64

Scenario 2 target:
c2_target
0    2434
1     586
Name: count, dtype: int64


In [16]:
# ============================================================
# STEP 135 — TRAIN ON SCENARIO 1
# TEST ON SCENARIO 2
# ============================================================

s1_split = int(len(s1) * 0.70)

s1_train = s1.iloc[:s1_split].copy()
s1_val = s1.iloc[s1_split:].copy()

s2_test = s2.copy()

X_s1_train = s1_train[common_features].copy()
y_s1_train = s1_train["c2_target"].copy()

X_s1_val = s1_val[common_features].copy()
y_s1_val = s1_val["c2_target"].copy()

X_s2_test = s2_test[common_features].copy()
y_s2_test = s2_test["c2_target"].copy()

print("Scenario 1 training:", X_s1_train.shape)
print("Scenario 1 validation:", X_s1_val.shape)
print("Scenario 2 unseen test:", X_s2_test.shape)

print("\nS1 train target:")
print(y_s1_train.value_counts())

print("\nS1 validation target:")
print(y_s1_val.value_counts())

print("\nS2 unseen test target:")
print(y_s2_test.value_counts())

Scenario 1 training: (3085, 19)
Scenario 1 validation: (1323, 19)
Scenario 2 unseen test: (3020, 19)

S1 train target:
c2_target
0    2860
1     225
Name: count, dtype: int64

S1 validation target:
c2_target
0    1242
1      81
Name: count, dtype: int64

S2 unseen test target:
c2_target
0    2434
1     586
Name: count, dtype: int64


In [17]:
# ============================================================
# STEP 136 — TRAINING-MEDIAN IMPUTATION
# ============================================================

s1_medians = (
    X_s1_train
    .median(numeric_only=True)
)

X_s1_train = X_s1_train.fillna(s1_medians)
X_s1_val = X_s1_val.fillna(s1_medians)
X_s2_test = X_s2_test.fillna(s1_medians)

print(
    "Remaining NaNs:",
    X_s1_train.isna().sum().sum(),
    X_s1_val.isna().sum().sum(),
    X_s2_test.isna().sum().sum()
)

Remaining NaNs: 0 0 0


In [18]:
# ============================================================
# STEP 137 — CROSS-SCENARIO HGB
# ============================================================

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier

s1_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_s1_train
)

c2_cross_s1_model = HistGradientBoostingClassifier(
    max_iter=350,
    learning_rate=0.04,
    max_leaf_nodes=15,
    min_samples_leaf=10,
    l2_regularization=2.0,
    random_state=42
)

c2_cross_s1_model.fit(
    X_s1_train,
    y_s1_train,
    sample_weight=s1_weights
)

s1_val_proba = (
    c2_cross_s1_model
    .predict_proba(X_s1_val)[:, 1]
)

print("Scenario 1 cross-scenario model trained")

c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Scenario 1 cross-scenario model trained


In [19]:
# ============================================================
# FIX — IMPORT METRICS FOR CROSS-SCENARIO EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("✅ Cross-scenario metrics imported")

✅ Cross-scenario metrics imported


In [20]:
# ============================================================
# STEP 138 — S1 VALIDATION THRESHOLD
# ============================================================

threshold_rows = []

for threshold in np.arange(
    0.10,
    0.91,
    0.02
):

    pred = (
        s1_val_proba >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": round(
            float(threshold),
            2
        ),
        "precision": precision_score(
            y_s1_val,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_s1_val,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_s1_val,
            pred,
            zero_division=0
        )
    })

s1_thresholds = pd.DataFrame(
    threshold_rows
)

print("S1 validation thresholds:")

display(
    s1_thresholds
    .sort_values(
        "f1",
        ascending=False
    )
    .head(10)
)

S1 validation thresholds:


,threshold,precision,recall,f1
2,0.14,0.071749,0.197531,0.105263
3,0.16,0.070652,0.160494,0.098113
1,0.12,0.058824,0.209877,0.091892
4,0.18,0.067568,0.123457,0.087336
0,0.10,0.052478,0.222222,0.084906
11,0.32,0.100000,0.061728,0.076336
10,0.30,0.083333,0.061728,0.070922
7,0.24,0.063830,0.074074,0.068571
14,0.38,0.111111,0.049383,0.068376
13,0.36,0.105263,0.049383,0.067227


In [21]:
# ============================================================
# STEP 139 — SELECT S1 THRESHOLD
# ============================================================

eligible_s1 = s1_thresholds[
    s1_thresholds["recall"] >= 0.70
]

if len(eligible_s1) > 0:

    best_s1 = (
        eligible_s1
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

else:

    best_s1 = (
        s1_thresholds
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

C2_CROSS_S1_THRESHOLD = float(
    best_s1["threshold"]
)

print(
    "Selected S1 threshold:",
    C2_CROSS_S1_THRESHOLD
)

print(
    "Validation precision:",
    best_s1["precision"]
)

print(
    "Validation recall:",
    best_s1["recall"]
)

print(
    "Validation F1:",
    best_s1["f1"]
)

Selected S1 threshold: 0.14
Validation precision: 0.07174887892376682
Validation recall: 0.19753086419753085
Validation F1: 0.10526315789473684


In [22]:
# ============================================================
# STEP 140 — SCENARIO 2 UNSEEN TEST
# ============================================================

s2_test_proba = (
    c2_cross_s1_model
    .predict_proba(X_s2_test)[:, 1]
)

s2_test_pred = (
    s2_test_proba >= C2_CROSS_S1_THRESHOLD
).astype(int)

print("TRAIN: SCENARIO 1")
print("TEST:  SCENARIO 2")

print(
    "Threshold:",
    C2_CROSS_S1_THRESHOLD
)

print(
    "Accuracy:",
    f"{accuracy_score(y_s2_test, s2_test_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_s2_test, s2_test_pred, zero_division=0):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_s2_test, s2_test_pred, zero_division=0):.4f}"
)

print(
    "F1:",
    f"{f1_score(y_s2_test, s2_test_pred, zero_division=0):.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_s2_test,
        s2_test_pred
    )
)

TRAIN: SCENARIO 1
TEST:  SCENARIO 2
Threshold: 0.14
Accuracy: 0.6503
Precision: 0.2664
Recall: 0.4573
F1: 0.3367

Confusion Matrix:
[[1696  738]
 [ 318  268]]


In [23]:
# ============================================================
# STEP 141 — TRAIN SCENARIO 2, TEST SCENARIO 1
# ============================================================

s2_split = int(len(s2) * 0.70)

s2_train = s2.iloc[:s2_split].copy()
s2_val = s2.iloc[s2_split:].copy()

s1_test = s1.copy()

X_s2_train = s2_train[common_features].copy()
y_s2_train = s2_train["c2_target"].copy()

X_s2_val = s2_val[common_features].copy()
y_s2_val = s2_val["c2_target"].copy()

X_s1_test = s1_test[common_features].copy()
y_s1_test = s1_test["c2_target"].copy()

s2_medians = (
    X_s2_train
    .median(numeric_only=True)
)

X_s2_train = X_s2_train.fillna(s2_medians)
X_s2_val = X_s2_val.fillna(s2_medians)
X_s1_test = X_s1_test.fillna(s2_medians)

s2_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_s2_train
)

c2_cross_s2_model = HistGradientBoostingClassifier(
    max_iter=350,
    learning_rate=0.04,
    max_leaf_nodes=15,
    min_samples_leaf=10,
    l2_regularization=2.0,
    random_state=42
)

c2_cross_s2_model.fit(
    X_s2_train,
    y_s2_train,
    sample_weight=s2_weights
)

s2_val_proba = (
    c2_cross_s2_model
    .predict_proba(X_s2_val)[:, 1]
)

print("Reverse cross-scenario model trained")

Reverse cross-scenario model trained


In [24]:
# ============================================================
# STEP 142 — S2 VALIDATION THRESHOLD
# ============================================================

reverse_threshold_rows = []

for threshold in np.arange(
    0.10,
    0.91,
    0.02
):

    pred = (
        s2_val_proba >= threshold
    ).astype(int)

    reverse_threshold_rows.append({
        "threshold": round(float(threshold), 2),

        "precision": precision_score(
            y_s2_val,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_s2_val,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_s2_val,
            pred,
            zero_division=0
        )
    })

s2_thresholds = pd.DataFrame(
    reverse_threshold_rows
)

eligible_s2 = s2_thresholds[
    s2_thresholds["recall"] >= 0.70
]

if len(eligible_s2) > 0:

    best_s2 = (
        eligible_s2
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

else:

    best_s2 = (
        s2_thresholds
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

C2_CROSS_S2_THRESHOLD = float(
    best_s2["threshold"]
)

print(
    "Selected S2 threshold:",
    C2_CROSS_S2_THRESHOLD
)

print(
    "Validation precision:",
    best_s2["precision"]
)

print(
    "Validation recall:",
    best_s2["recall"]
)

print(
    "Validation F1:",
    best_s2["f1"]
)

Selected S2 threshold: 0.1
Validation precision: 0.165402124430956
Validation recall: 0.7465753424657534
Validation F1: 0.2708074534161491


In [25]:
# ============================================================
# STEP 143 — SCENARIO 1 UNSEEN TEST
# ============================================================

s1_test_proba = (
    c2_cross_s2_model
    .predict_proba(X_s1_test)[:, 1]
)

s1_test_pred = (
    s1_test_proba >= C2_CROSS_S2_THRESHOLD
).astype(int)

print("TRAIN: SCENARIO 2")
print("TEST:  SCENARIO 1")

print(
    "Threshold:",
    C2_CROSS_S2_THRESHOLD
)

print(
    "Accuracy:",
    f"{accuracy_score(y_s1_test, s1_test_pred):.4f}"
)

print(
    "Precision:",
    f"{precision_score(y_s1_test, s1_test_pred, zero_division=0):.4f}"
)

print(
    "Recall:",
    f"{recall_score(y_s1_test, s1_test_pred, zero_division=0):.4f}"
)

print(
    "F1:",
    f"{f1_score(y_s1_test, s1_test_pred, zero_division=0):.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_s1_test,
        s1_test_pred
    )
)

TRAIN: SCENARIO 2
TEST:  SCENARIO 1
Threshold: 0.1
Accuracy: 0.3437
Precision: 0.0807
Recall: 0.8137
F1: 0.1469

Confusion Matrix:
[[1266 2836]
 [  57  249]]


In [26]:
# ============================================================
# STEP 144 — CROSS-SCENARIO SUMMARY
# ============================================================

cross_scenario_results = pd.DataFrame([
    {
        "Train": "CTU13_42",
        "Test": "CTU13_43",
        "Accuracy": accuracy_score(
            y_s2_test,
            s2_test_pred
        ),
        "Precision": precision_score(
            y_s2_test,
            s2_test_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_s2_test,
            s2_test_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_s2_test,
            s2_test_pred,
            zero_division=0
        )
    },
    {
        "Train": "CTU13_43",
        "Test": "CTU13_42",
        "Accuracy": accuracy_score(
            y_s1_test,
            s1_test_pred
        ),
        "Precision": precision_score(
            y_s1_test,
            s1_test_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_s1_test,
            s1_test_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_s1_test,
            s1_test_pred,
            zero_division=0
        )
    }
])

display(
    cross_scenario_results
)

,Train,Test,Accuracy,Precision,Recall,F1
0,CTU13_42,CTU13_43,0.650331,0.266402,0.457338,0.336683
1,CTU13_43,CTU13_42,0.343693,0.080713,0.813725,0.146859


In [27]:
# ============================================================
# STEP 145 — LOAD CTU-13 SCENARIO 3
# ============================================================

SCENARIO3_FILE = Path(
    "../data/raw/CTU-13/scenario3/capture20110812.binetflow"
)

scenario3_columns = [
    "StartTime",
    "Dur",
    "Proto",
    "SrcAddr",
    "Sport",
    "Dir",
    "DstAddr",
    "Dport",
    "TotPkts",
    "TotBytes",
    "SrcBytes",
    "Label"
]

c2_s3 = pd.read_csv(
    SCENARIO3_FILE,
    usecols=scenario3_columns
)

print("Scenario 3 shape:", c2_s3.shape)
print("Columns:", c2_s3.columns.tolist())

Scenario 3 shape: (4710638, 12)
Columns: ['StartTime', 'Dur', 'Proto', 'SrcAddr', 'Sport', 'Dir', 'DstAddr', 'Dport', 'TotPkts', 'TotBytes', 'SrcBytes', 'Label']


In [28]:
# ============================================================
# STEP 146 — SCENARIO 3 LABEL ANALYSIS
# ============================================================

print("Total flows:", len(c2_s3))
print("Unique labels:", c2_s3["Label"].nunique())

print("\nTop labels:")

display(
    c2_s3["Label"]
    .value_counts()
    .head(40)
)

c2_s3_cc = c2_s3[
    c2_s3["Label"]
    .astype(str)
    .str.contains(
        r"-CC\d+-",
        case=False,
        regex=True
    )
].copy()

print("\nExplicit C&C flows:", len(c2_s3_cc))

print("\nC&C labels:")

display(
    c2_s3_cc["Label"]
    .value_counts()
)

Total flows: 4710638
Unique labels: 51

Top labels:


Label
flow=To-Background-UDP-CVUT-DNS-Server    2169040
flow=Background-UDP-Established            806119
flow=Background-TCP-Established            729048
flow=Background-TCP-Attempt                408887
flow=Background                            156122
flow=From-Normal-V44-Stribrek              108807
flow=Background-UDP-Attempt                104089
flow=Background-Established-cmpgw-CVUT      85383
flow=From-Botnet-V44-TCP-Attempt            26234
flow=To-Background-CVUT-WebServer           24271
flow=Background-UDP-NTP-Established-1       22345
flow=To-Background-CVUT-Proxy               22009
flow=Background-Attempt-cmpgw-CVUT          13659
flow=To-Background-MatLab-Server             8937
flow=From-Normal-V44-Grill                   4580
flow=Background-google-pop                   2654
flow=From-Normal-V44-CVUT-WebServer          1843
flow=Background-google-webmail               1562
flow=From-Background-CVUT-Proxy              1041
flow=From-Normal-V44-Jist                   


Explicit C&C flows: 63

C&C labels:


Label
flow=From-Botnet-V44-TCP-CC107-IRC-Not-Encrypted    63
Name: count, dtype: int64

In [29]:
# ============================================================
# STEP 147 — SCENARIO 3 C&C SOURCE ANALYSIS
# ============================================================

print("Top C&C source addresses:")

display(
    c2_s3_cc["SrcAddr"]
    .value_counts()
    .head(20)
)

print(
    "\nNumber of C&C source addresses:",
    c2_s3_cc["SrcAddr"].nunique()
)

Top C&C source addresses:


SrcAddr
38.229.70.20    63
Name: count, dtype: int64


Number of C&C source addresses: 1


In [30]:
# ============================================================
# STEP 148 — BUILD SCENARIO 3 C2 FEATURES
# ============================================================

scenario3_c2_features = build_c2_window_features(
    c2_s3,
    window_seconds=5
)

print(
    "Scenario 3 feature table:",
    scenario3_c2_features.shape
)

print("\nScenario 3 target distribution:")

print(
    scenario3_c2_features[
        "c2_target"
    ].value_counts()
)

Scenario 3 feature table: (48114, 21)

Scenario 3 target distribution:
c2_target
0    48051
1       63
Name: count, dtype: int64


In [31]:
# ============================================================
# STEP 149 — SCENARIO 3 IDENTIFIER
# ============================================================

scenario3_c2_features["scenario"] = "CTU13_44"

print(
    scenario3_c2_features["scenario"].unique()
)

['CTU13_44']


In [32]:
# ============================================================
# STEP 150 — THREE-SCENARIO SUMMARY
# ============================================================

scenario_datasets = {
    "CTU13_42": scenario1_c2_features,
    "CTU13_43": scenario2_c2_features,
    "CTU13_44": scenario3_c2_features
}

for name, data in scenario_datasets.items():

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print("Shape:", data.shape)

    print("\nTarget distribution:")

    print(
        data["c2_target"].value_counts()
    )

    print(
        "\nC2 percentage:",
        round(
            data["c2_target"].mean() * 100,
            2
        )
    )


CTU13_42
Shape: (4408, 22)

Target distribution:
c2_target
0    4102
1     306
Name: count, dtype: int64

C2 percentage: 6.94

CTU13_43
Shape: (3020, 22)

Target distribution:
c2_target
0    2434
1     586
Name: count, dtype: int64

C2 percentage: 19.4

CTU13_44
Shape: (48114, 22)

Target distribution:
c2_target
0    48051
1       63
Name: count, dtype: int64

C2 percentage: 0.13


In [33]:
# ============================================================
# STEP 151 — THREE-SCENARIO DATA PREPARATION
# ============================================================

scenario1 = (
    scenario1_c2_features
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

scenario2 = (
    scenario2_c2_features
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

scenario3 = (
    scenario3_c2_features
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

# Keep only the 19 features shared by every scenario.
common_features = sorted(
    list(
        (
            set(scenario1.columns)
            & set(scenario2.columns)
            & set(scenario3.columns)
        )
        - {
            "time_window",
            "c2_target",
            "scenario"
        }
    )
)

print("Common feature count:", len(common_features))

print("\nScenario sizes:")
print("CTU13_42:", scenario1.shape)
print("CTU13_43:", scenario2.shape)
print("CTU13_44:", scenario3.shape)

Common feature count: 19

Scenario sizes:
CTU13_42: (4408, 22)
CTU13_43: (3020, 22)
CTU13_44: (48114, 22)


In [34]:
# ============================================================
# STEP 152 — FEATURE CONSISTENCY CHECK
# ============================================================

print("Feature columns are identical across scenarios:")

print(
    set(common_features)
    == (
        set(scenario1.columns)
        - {"time_window", "c2_target", "scenario"}
    ).intersection(
        set(scenario2.columns)
        - {"time_window", "c2_target", "scenario"}
    ).intersection(
        set(scenario3.columns)
        - {"time_window", "c2_target", "scenario"}
    )
)

print("\nNumber of common features:", len(common_features))

Feature columns are identical across scenarios:
True

Number of common features: 19


In [35]:
# ============================================================
# STEP 153 — LEAVE-ONE-SCENARIO-OUT EVALUATION FUNCTION
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def run_leave_one_scenario_out(
    train_scenarios,
    test_scenario,
    feature_columns,
    validation_fraction=0.30
):

    # --------------------------------------------------------
    # Combine training scenarios
    # --------------------------------------------------------

    train_data = pd.concat(
        train_scenarios,
        ignore_index=True
    )

    train_data = (
        train_data
        .sort_values("time_window")
        .reset_index(drop=True)
    )

    test_data = (
        test_scenario
        .sort_values("time_window")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Temporal validation split inside training data
    # --------------------------------------------------------

    split_point = int(
        len(train_data) *
        (1 - validation_fraction)
    )

    train_part = train_data.iloc[
        :split_point
    ].copy()

    val_part = train_data.iloc[
        split_point:
    ].copy()

    # --------------------------------------------------------
    # X / y
    # --------------------------------------------------------

    X_train = train_part[
        feature_columns
    ].copy()

    y_train = train_part[
        "c2_target"
    ].copy()

    X_val = val_part[
        feature_columns
    ].copy()

    y_val = val_part[
        "c2_target"
    ].copy()

    X_test = test_data[
        feature_columns
    ].copy()

    y_test = test_data[
        "c2_target"
    ].copy()

    # --------------------------------------------------------
    # Median imputation using training only
    # --------------------------------------------------------

    medians = (
        X_train
        .median(numeric_only=True)
    )

    X_train = X_train.fillna(medians)
    X_val = X_val.fillna(medians)
    X_test = X_test.fillna(medians)

    # --------------------------------------------------------
    # Balanced sample weights
    # --------------------------------------------------------

    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.04,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=2.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        sample_weight=sample_weights
    )

    # --------------------------------------------------------
    # Validation probabilities
    # --------------------------------------------------------

    val_proba = (
        model
        .predict_proba(X_val)[:, 1]
    )

    # --------------------------------------------------------
    # Threshold search on validation ONLY
    # --------------------------------------------------------

    threshold_rows = []

    for threshold in np.arange(
        0.05,
        0.96,
        0.02
    ):

        val_pred = (
            val_proba >= threshold
        ).astype(int)

        precision = precision_score(
            y_val,
            val_pred,
            zero_division=0
        )

        recall = recall_score(
            y_val,
            val_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_val,
            val_pred,
            zero_division=0
        )

        threshold_rows.append({
            "threshold": round(
                float(threshold),
                2
            ),
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    # Prefer useful recall while maximizing F1.
    eligible = threshold_table[
        threshold_table["recall"] >= 0.70
    ]

    if len(eligible) > 0:

        best_row = (
            eligible
            .sort_values(
                "f1",
                ascending=False
            )
            .iloc[0]
        )

    else:

        best_row = (
            threshold_table
            .sort_values(
                "f1",
                ascending=False
            )
            .iloc[0]
        )

    threshold = float(
        best_row["threshold"]
    )

    # --------------------------------------------------------
    # Completely unseen scenario
    # --------------------------------------------------------

    test_proba = (
        model
        .predict_proba(X_test)[:, 1]
    )

    test_pred = (
        test_proba >= threshold
    ).astype(int)

    metrics = {
        "Accuracy": accuracy_score(
            y_test,
            test_pred
        ),
        "Precision": precision_score(
            y_test,
            test_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            test_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            test_pred,
            zero_division=0
        )
    }

    return {
        "model": model,
        "threshold": threshold,
        "validation": best_row,
        "test_metrics": metrics,
        "confusion_matrix": confusion_matrix(
            y_test,
            test_pred
        ),
        "threshold_table": threshold_table,
        "test_probabilities": test_proba,
        "test_predictions": test_pred,
        "y_test": y_test,
        "feature_columns": feature_columns
    }

In [36]:
# ============================================================
# STEP 154 — TRAIN 42+43 → TEST 44
# ============================================================

result_44 = run_leave_one_scenario_out(
    train_scenarios=[
        scenario1,
        scenario2
    ],
    test_scenario=scenario3,
    feature_columns=common_features
)


print("TRAIN: CTU13_42 + CTU13_43")
print("TEST:  CTU13_44")

print(
    "\nSelected threshold:",
    result_44["threshold"]
)

print("\nValidation:")
print(
    result_44["validation"]
)

print("\nUnseen Scenario 44 Test:")

for name, value in result_44[
    "test_metrics"
].items():

    print(
        f"{name:10s}: {value:.4f}"
    )

print("\nConfusion Matrix:")
print(
    result_44["confusion_matrix"]
)

TRAIN: CTU13_42 + CTU13_43
TEST:  CTU13_44

Selected threshold: 0.07

Validation:
threshold    0.070000
precision    0.240521
recall       0.823529
f1           0.372306
Name: 1, dtype: float64

Unseen Scenario 44 Test:
Accuracy  : 0.9371
Precision : 0.0003
Recall    : 0.0159
F1        : 0.0007

Confusion Matrix:
[[45088  2963]
 [   62     1]]


In [37]:
# ============================================================
# STEP 155 — TRAIN 42+44 → TEST 43
# ============================================================

result_43 = run_leave_one_scenario_out(
    train_scenarios=[
        scenario1,
        scenario3
    ],
    test_scenario=scenario2,
    feature_columns=common_features
)


print("TRAIN: CTU13_42 + CTU13_44")
print("TEST:  CTU13_43")

print(
    "\nSelected threshold:",
    result_43["threshold"]
)

print("\nValidation:")
print(
    result_43["validation"]
)

print("\nUnseen Scenario 43 Test:")

for name, value in result_43[
    "test_metrics"
].items():

    print(
        f"{name:10s}: {value:.4f}"
    )

print("\nConfusion Matrix:")
print(
    result_43["confusion_matrix"]
)

TRAIN: CTU13_42 + CTU13_44
TEST:  CTU13_43

Selected threshold: 0.15

Validation:
threshold    0.150000
precision    0.001476
recall       0.736842
f1           0.002945
Name: 5, dtype: float64

Unseen Scenario 43 Test:
Accuracy  : 0.4838
Precision : 0.2163
Recall    : 0.6331
F1        : 0.3225

Confusion Matrix:
[[1090 1344]
 [ 215  371]]


In [38]:
# ============================================================
# STEP 156 — TRAIN 43+44 → TEST 42
# ============================================================

result_42 = run_leave_one_scenario_out(
    train_scenarios=[
        scenario2,
        scenario3
    ],
    test_scenario=scenario1,
    feature_columns=common_features
)


print("TRAIN: CTU13_43 + CTU13_44")
print("TEST:  CTU13_42")

print(
    "\nSelected threshold:",
    result_42["threshold"]
)

print("\nValidation:")
print(
    result_42["validation"]
)

print("\nUnseen Scenario 42 Test:")

for name, value in result_42[
    "test_metrics"
].items():

    print(
        f"{name:10s}: {value:.4f}"
    )

print("\nConfusion Matrix:")
print(
    result_42["confusion_matrix"]
)

TRAIN: CTU13_43 + CTU13_44
TEST:  CTU13_42

Selected threshold: 0.05

Validation:
threshold    0.050000
precision    0.001465
recall       0.888889
f1           0.002925
Name: 0, dtype: float64

Unseen Scenario 42 Test:
Accuracy  : 0.1078
Precision : 0.0722
Recall    : 1.0000
F1        : 0.1347

Confusion Matrix:
[[ 169 3933]
 [   0  306]]


In [39]:
# ============================================================
# STEP 157 — FINAL CROSS-SCENARIO SUMMARY
# ============================================================

cross_results = pd.DataFrame([
    {
        "Train": "CTU13_42 + CTU13_43",
        "Test": "CTU13_44",
        **result_44["test_metrics"]
    },
    {
        "Train": "CTU13_42 + CTU13_44",
        "Test": "CTU13_43",
        **result_43["test_metrics"]
    },
    {
        "Train": "CTU13_43 + CTU13_44",
        "Test": "CTU13_42",
        **result_42["test_metrics"]
    }
])

display(
    cross_results
)

,Train,Test,Accuracy,Precision,Recall,F1
0,CTU13_42 + CTU13_43,CTU13_44,0.937128,0.000337,0.015873,0.000661
1,CTU13_42 + CTU13_44,CTU13_43,0.483775,0.216327,0.633106,0.322468
2,CTU13_43 + CTU13_44,CTU13_42,0.107759,0.072187,1.000000,0.134653


In [40]:
# ============================================================
# STEP 158 — CROSS-SCENARIO AVERAGE
# ============================================================

print("Mean cross-scenario performance:")

display(
    cross_results[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1"
        ]
    ].mean()
)

Mean cross-scenario performance:


Accuracy     0.509554
Precision    0.096284
Recall       0.549660
F1           0.152594
dtype: float64

In [41]:
# ============================================================
# STEP 159 — SCENARIO 3 C2 DETECTION COUNT
# ============================================================

test_y = result_44["y_test"]
test_pred = result_44["test_predictions"]

actual_c2 = int(
    (test_y == 1).sum()
)

detected_c2 = int(
    (
        (test_y == 1)
        &
        (test_pred == 1)
    ).sum()
)

missed_c2 = int(
    (
        (test_y == 1)
        &
        (test_pred == 0)
    ).sum()
)

print("Scenario 3 actual C2 windows:", actual_c2)
print("Detected C2 windows:", detected_c2)
print("Missed C2 windows:", missed_c2)

Scenario 3 actual C2 windows: 63
Detected C2 windows: 1
Missed C2 windows: 62


In [42]:
# ============================================================
# STEP 160 — HYBRID C2 BEHAVIOUR SCORE
# ============================================================

def calculate_c2_behaviour_score(df):
    """
    Rule-based behavioural score using only observable
    flow/window characteristics.

    Higher score = stronger C2-like behaviour.
    """

    data = df.copy()

    score = pd.Series(
        0.0,
        index=data.index
    )

    # --------------------------------------------------------
    # 1. Repeated destination behaviour
    # --------------------------------------------------------

    if "max_pair_repetition" in data.columns:

        score += np.clip(
            data["max_pair_repetition"] / 5.0,
            0,
            1
        ) * 0.20

    # --------------------------------------------------------
    # 2. Repeated pair ratio
    # --------------------------------------------------------

    if "pair_repetition_ratio" in data.columns:

        score += np.clip(
            data["pair_repetition_ratio"],
            0,
            1
        ) * 0.20

    # --------------------------------------------------------
    # 3. Destination concentration
    # --------------------------------------------------------

    if "destination_concentration" in data.columns:

        score += np.clip(
            data["destination_concentration"],
            0,
            1
        ) * 0.15

    # --------------------------------------------------------
    # 4. Recent pair contacts
    # --------------------------------------------------------

    if "max_recent_contacts_60s" in data.columns:

        score += np.clip(
            data["max_recent_contacts_60s"] / 10.0,
            0,
            1
        ) * 0.15

    # --------------------------------------------------------
    # 5. Longer-term pair contacts
    # --------------------------------------------------------

    if "max_recent_contacts_300s" in data.columns:

        score += np.clip(
            data["max_recent_contacts_300s"] / 30.0,
            0,
            1
        ) * 0.10

    # --------------------------------------------------------
    # 6. IAT stability
    # --------------------------------------------------------

    if "iat_cv" in data.columns:

        iat_stability = 1 / (
            1 + data["iat_cv"].fillna(999)
        )

        score += np.clip(
            iat_stability,
            0,
            1
        ) * 0.10

    # --------------------------------------------------------
    # 7. Destination entropy
    # --------------------------------------------------------

    if "destination_entropy" in data.columns:

        entropy_score = 1 / (
            1 + data["destination_entropy"].fillna(999)
        )

        score += np.clip(
            entropy_score,
            0,
            1
        ) * 0.10

    return score.clip(0, 1)

In [43]:
# ============================================================
# STEP 161 — BEHAVIOURAL SCORES
# ============================================================

scenario1["behaviour_score"] = (
    calculate_c2_behaviour_score(
        scenario1
    )
)

scenario2["behaviour_score"] = (
    calculate_c2_behaviour_score(
        scenario2
    )
)

scenario3["behaviour_score"] = (
    calculate_c2_behaviour_score(
        scenario3
    )
)

print("Behaviour scores generated")

Behaviour scores generated


In [44]:
# ============================================================
# STEP 162 — BEHAVIOURAL SCORE BY CLASS
# ============================================================

for name, data in [
    ("Scenario 1", scenario1),
    ("Scenario 2", scenario2),
    ("Scenario 3", scenario3)
]:

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    display(
        data
        .groupby("c2_target")["behaviour_score"]
        .describe()
    )


Scenario 1


,count,mean,std,min,25%,50%,75%,max
c2_target,,,,,,,,
0,4102.0,0.388511,0.021845,0.250463,0.374928,0.387236,0.399761,0.536635
1,306.0,0.389056,0.017612,0.306233,0.378162,0.388369,0.400978,0.439756



Scenario 2


,count,mean,std,min,25%,50%,75%,max
c2_target,,,,,,,,
0,2434.0,0.389395,0.027844,0.276626,0.372509,0.387626,0.403483,0.529537
1,586.0,0.383671,0.028808,0.305793,0.368318,0.380525,0.396041,0.513525



Scenario 3


,count,mean,std,min,25%,50%,75%,max
c2_target,,,,,,,,
0,48051.0,0.424975,0.052491,0.231598,0.384810,0.418298,0.460490,0.638417
1,63.0,0.421714,0.058024,0.325929,0.375142,0.412359,0.459247,0.571617


In [45]:
# ============================================================
# STEP 163 — SCENARIO 3 C2 SCORES
# ============================================================

s3_c2 = scenario3[
    scenario3["c2_target"] == 1
].copy()

print(
    "Scenario 3 C2 windows:",
    len(s3_c2)
)

display(
    s3_c2[
        [
            "time_window",
            "behaviour_score",
            "flow_count",
            "unique_destinations",
            "unique_ports",
            "total_packets",
            "total_bytes",
            "destination_concentration",
            "iat_mean",
            "iat_cv",
            "max_pair_repetition",
            "pair_repetition_ratio",
            "destination_entropy"
        ]
    ]
    .sort_values(
        "behaviour_score",
        ascending=False
    )
    .head(30)
)

Scenario 3 C2 windows: 63


,time_window,behaviour_score,flow_count,unique_destinations,unique_ports,total_packets,total_bytes,destination_concentration,iat_mean,iat_cv,max_pair_repetition,pair_repetition_ratio,destination_entropy
11177,2011-08-13 06:55:25,0.571617,166,13,10,930,507549,0.885542,29.371743,10.105842,147,0.885542,0.898493
5969,2011-08-12 23:41:25,0.559817,165,18,9,804,275079,0.866667,28.165400,10.399703,143,0.866667,1.095919
17874,2011-08-13 16:13:30,0.558497,164,16,8,2224,320833,0.859756,81.979304,9.149857,141,0.859756,1.095105
26035,2011-08-14 03:33:35,0.545002,169,23,8,1008,406638,0.834320,249.483580,8.281949,141,0.834320,1.368740
9686,2011-08-13 04:51:10,0.510438,79,15,7,1678,355981,0.746835,103.152554,5.795635,58,0.734177,1.712824
43101,2011-08-15 03:15:45,0.508806,215,17,10,5198,1009825,0.739535,41.537985,10.156070,159,0.739535,1.438714
5224,2011-08-12 22:39:20,0.484172,48,14,8,451,147516,0.666667,794.320109,4.349358,32,0.666667,2.110902
12667,2011-08-13 08:59:35,0.483574,51,14,10,494,231764,0.647059,512.160352,2.957019,33,0.647059,2.141550
27525,2011-08-14 05:37:45,0.479966,61,17,8,563,245578,0.655738,256.868902,4.043715,40,0.655738,2.264639
14148,2011-08-13 11:03:00,0.474883,55,16,9,1990,845872,0.636364,256.934984,3.604475,35,0.636364,2.285363


In [46]:
# ============================================================
# STEP 164 — C2 BEHAVIOUR THRESHOLD SEARCH
# ============================================================

threshold_rows = []

for threshold in np.arange(
    0.05,
    0.96,
    0.05
):

    for name, data in [
        ("Scenario1", scenario1),
        ("Scenario2", scenario2),
        ("Scenario3", scenario3)
    ]:

        pred = (
            data["behaviour_score"]
            >= threshold
        ).astype(int)

        threshold_rows.append({
            "Scenario": name,
            "Threshold": threshold,
            "Precision": precision_score(
                data["c2_target"],
                pred,
                zero_division=0
            ),
            "Recall": recall_score(
                data["c2_target"],
                pred,
                zero_division=0
            ),
            "F1": f1_score(
                data["c2_target"],
                pred,
                zero_division=0
            )
        })

behaviour_thresholds = pd.DataFrame(
    threshold_rows
)

display(
    behaviour_thresholds
    .sort_values(
        ["Scenario", "F1"],
        ascending=[True, False]
    )
)

,Scenario,Threshold,Precision,Recall,F1
18,Scenario1,0.35,0.069741,0.983660,0.130247
15,Scenario1,0.30,0.069514,1.000000,0.129992
0,Scenario1,0.05,0.069419,1.000000,0.129826
3,Scenario1,0.10,0.069419,1.000000,0.129826
6,Scenario1,0.15,0.069419,1.000000,0.129826
9,Scenario1,0.20,0.069419,1.000000,0.129826
12,Scenario1,0.25,0.069419,1.000000,0.129826
21,Scenario1,0.40,0.075592,0.271242,0.118234
24,Scenario1,0.45,0.000000,0.000000,0.000000
27,Scenario1,0.50,0.000000,0.000000,0.000000


In [49]:
# ============================================================
# FIX — ADD PROJECT ROOT TO PYTHON PATH
# ============================================================

from pathlib import Path
import sys

# Find the project directory containing the src folder.
current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing 'src' folder."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src exists:", (project_root / "src").is_dir())

Project root: e:\CODEZILLA-SIH26145
src exists: True


In [50]:
import pandas as pd

from src.detectors.c2_detector import detect


test_window = pd.DataFrame([
    {
        "destination_concentration": 0.75,
        "pair_repetition_ratio": 0.60,
        "max_pair_repetition": 7,
        "max_recent_contacts_60s": 6,
        "max_recent_contacts_300s": 20,
        "iat_cv": 0.45,
    }
])

result = detect(
    test_window,
    top_k=5
)

print(result)

[{'prediction': 'SUSPICIOUS', 'model_score': 0.6129, 'decision_threshold': 0.5, 'threat_class': 'C2', 'severity': 'MEDIUM', 'supporting_features': [{'signal': 'destination_concentration', 'value': 0.75, 'contribution': 0.15000000000000002}, {'signal': 'communication_periodicity', 'value': 0.45, 'contribution': 0.13793103448275865}, {'signal': 'pair_repetition', 'value': 0.6, 'contribution': 0.12}, {'signal': 'repeated_pair_count', 'value': 7.0, 'contribution': 0.105}, {'signal': 'recent_repeated_contact', 'value': 20.0, 'contribution': 0.09999999999999999}]}]


In [51]:
import pandas as pd

from src.detectors.dns_detector import detect

dns_test = pd.DataFrame([
    {
        "dns_port_activity": 1.0,
        "dns_query_rate": 25,
        "dns_unique_destinations": 12,
        "dns_packet_concentration": 0.75,
        "dns_byte_concentration": 0.65,
        "dns_iat_cv": 0.40,
    }
])

result = detect(
    dns_test,
    top_k=5
)

print(result)

[{'prediction': 'DNS_THREAT', 'model_score': 0.8246, 'decision_threshold': 0.5, 'threat_class': 'DNS', 'severity': 'HIGH', 'supporting_features': [{'signal': 'dns_port_activity', 'value': 1.0, 'contribution': 0.25}, {'signal': 'dns_query_rate', 'value': 25.0, 'contribution': 0.2}, {'signal': 'dns_packet_concentration', 'value': 0.75, 'contribution': 0.11249999999999999}, {'signal': 'dns_timing_pattern', 'value': 0.4, 'contribution': 0.10714285714285714}, {'signal': 'dns_destination_diversity', 'value': 12.0, 'contribution': 0.09}]}]


In [52]:
# ============================================================
# STEP 262 — SAVE C2 FEATURE ARTIFACT
# ============================================================

from pathlib import Path
import pandas as pd

# Find project root
project_root = Path.cwd()

while not (project_root / "models").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root.")
    project_root = project_root.parent

# ------------------------------------------------------------
# Find the already-created C2 dataframe
# ------------------------------------------------------------

candidates = []

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        # C2 tables normally contain these
        score = 0

        if "time_window" in cols:
            score += 1

        if "c2_target" in cols:
            score += 2

        if "SrcAddr" in cols:
            score += 2

        if "flow_count" in cols:
            score += 1

        if "total_bytes" in cols:
            score += 1

        if score >= 4:
            candidates.append(
                (name, obj, score)
            )

print("Possible C2 feature tables:\n")

for name, obj, score in sorted(
    candidates,
    key=lambda x: x[2],
    reverse=True
):
    print(
        f"{name}: "
        f"shape={obj.shape}, "
        f"score={score}"
    )

if not candidates:
    raise RuntimeError(
        "No C2 feature dataframe found in this notebook."
    )

# Best candidate
c2_name, c2_df, _ = sorted(
    candidates,
    key=lambda x: x[2],
    reverse=True
)[0]

print(
    "\n✅ Selected C2 dataframe:",
    c2_name
)

print(
    "Shape:",
    c2_df.shape
)

print(
    "Columns:",
    list(c2_df.columns)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_dir = (
    project_root
    / "data"
    / "processed"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir
    / "c2_features.parquet"
)

c2_df.to_parquet(
    output_path,
    index=False
)

print(
    "\n✅ C2 feature artifact saved"
)

print(
    "Path:",
    output_path
)

Possible C2 feature tables:

scenario2_c2_features: shape=(3020, 22), score=5
scenario1_c2_features: shape=(4408, 22), score=5
data: shape=(48114, 23), score=5
s1: shape=(4408, 22), score=5
s2: shape=(3020, 22), score=5
s1_train: shape=(3085, 22), score=5
s1_val: shape=(1323, 22), score=5
s2_test: shape=(3020, 22), score=5
s2_train: shape=(2114, 22), score=5
s2_val: shape=(906, 22), score=5
s1_test: shape=(4408, 22), score=5
scenario3_c2_features: shape=(48114, 22), score=5
scenario1: shape=(4408, 23), score=5
scenario2: shape=(3020, 23), score=5
scenario3: shape=(48114, 23), score=5
s3_c2: shape=(63, 23), score=5

✅ Selected C2 dataframe: scenario2_c2_features
Shape: (3020, 22)
Columns: ['time_window', 'flow_count', 'unique_destinations', 'unique_ports', 'total_packets', 'total_bytes', 'mean_packets', 'mean_bytes', 'mean_duration', 'max_destination_count', 'destination_concentration', 'iat_mean', 'iat_std', 'iat_median', 'iat_min', 'iat_max', 'iat_cv', 'max_pair_repetition', 'pair_rep